In [ ]:
import os
import io
from PIL import Image
import pandas as pd
from datasets import Dataset, Features, ClassLabel, Image as DatasetImage
from huggingface_hub import HfApi, upload_file

# CONFIG
DATA_DIR = "./dataset"  # your dataset root directory
PARQUET_PATH = "train_dataset.parquet"
HF_DATASET_REPO = ""
HF_TOKEN = ""  # Consider using an env var instead

# STEP 1: Load images as bytes
data = []
class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

for label in class_names:
    class_dir = os.path.join(DATA_DIR, label)
    for fname in os.listdir(class_dir):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            image_path = os.path.join(class_dir, fname)
            try:
                with Image.open(image_path) as img:
                    img = img.convert("RGB")
                    buffer = io.BytesIO()
                    img.save(buffer, format="JPEG")
                    img_bytes = buffer.getvalue()
                    data.append({
                        "image": {"bytes": img_bytes, "path": None},
                        "label": label
                    })
            except Exception as e:
                print(f"⚠️ Failed to load {image_path}: {e}")

# STEP 2: Create Dataset
df = pd.DataFrame(data)

features = Features({
    "image": DatasetImage(),  # interprets {'bytes': ..., 'path': None}
    "label": ClassLabel(names=class_names)
})

dataset = Dataset.from_pandas(df, features=features)

# STEP 3: Save to Parquet
dataset.to_parquet(PARQUET_PATH)

# STEP 4: Upload to HF Hub
api = HfApi()
api.create_repo(repo_id=HF_DATASET_REPO, token=HF_TOKEN, repo_type="dataset", exist_ok=True)

upload_file(
    path_or_fileobj=PARQUET_PATH,
    path_in_repo="train_dataset.parquet",
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)

print(f"✅ Uploaded to: https://huggingface.co/datasets/{HF_DATASET_REPO}")


Creating parquet from Arrow format:   0%|          | 0/37 [00:00<?, ?ba/s]

train_dataset.parquet:   0%|          | 0.00/58.0M [00:00<?, ?B/s]

✅ Uploaded to: https://huggingface.co/datasets/Humayoun/Cancer


In [2]:
!pip install datasets

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
whe